# Vanity Metrics: The Numbers That Feel Good and Decide Nothing

Companion notebook for the article on the
[Marketing Data Science blog](https://blog.marketingdatascience.ai) by Joe Domaleski.

A vanity metric is a number that reliably goes up and never changes a decision.
This notebook builds 52 weeks of **simulated** marketing data where the follower
count grows 91% while revenue quietly falls by half, and then shows how to tell
the two kinds of metrics apart.

Everything here is simulated. No client data is used anywhere in this notebook.

**Run order:** top to bottom. Part 5 is the part you edit to test your own metrics.

## Part 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Clean, white-background figure style
mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#333333",
    "grid.color": "#DDDDDD",
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

VANITY = "#6E8CA0"    # muted steel blue
DECISION = "#B4552D"  # rust
DIP_SHADE = "#EFEFEF"

RNG = np.random.default_rng(42)
N_WEEKS = 52
print("Ready.")

## Part 2. Build a year of simulated data

The data generating process has three layers, and the whole point is in how they
connect.

**The vanity layer.** Followers only ever go up. That is the defining property.
Impressions track followers loosely.

**The traffic layer.** Sessions come mostly from paid spend and organic search,
only weakly from social impressions.

**The decision layer.** The qualified lead rate is what actually moves money, and
unlike followers it can go **down**. Here it collapses during weeks 17 to 31
because a form broke and the targeting drifted, then recovers.

That non-monotone dip is the entire experiment. Watch which charts notice it.

In [ ]:
week = np.arange(1, N_WEEKS + 1)
week_start = pd.date_range("2025-08-04", periods=N_WEEKS, freq="W-MON")

# --- Vanity layer: followers only ever go up -------------------------------
follower_gains = RNG.normal(loc=42, scale=14, size=N_WEEKS).clip(min=2)
followers = (2400 + np.cumsum(follower_gains)).round().astype(int)

posting_boost = RNG.normal(0, 1, N_WEEKS)
impressions = (18000 + 7.5 * followers + 9000 * posting_boost
               + RNG.normal(0, 4000, N_WEEKS)).clip(min=5000).round().astype(int)

# --- Traffic layer ---------------------------------------------------------
paid_spend = (2200 + 550 * np.sin(week / 4.0)
              + RNG.normal(0, 340, N_WEEKS)).clip(min=600).round(2)

sessions = (650 + 0.0035 * impressions + 0.22 * paid_spend
            + RNG.normal(0, 95, N_WEEKS)).clip(min=200).round().astype(int)

# --- Decision layer: the rate that can actually fall -----------------------
base_rate = np.full(N_WEEKS, 0.0385)
dip = (week >= 18) & (week <= 30)
ramp_down = np.clip((week - 15) / 4.0, 0, 1)
ramp_up = np.clip((34 - week) / 4.0, 0, 1)
dip_depth = np.clip(np.where(dip, 1.0, np.minimum(ramp_down, ramp_up)), 0, 1)
base_rate = base_rate - 0.0175 * dip_depth

qualified_lead_rate = (base_rate + RNG.normal(0, 0.0022, N_WEEKS)).clip(min=0.004)
qualified_leads = np.maximum(0, RNG.poisson(sessions * qualified_lead_rate)).astype(int)

revenue_per_lead = RNG.normal(465, 52, N_WEEKS).clip(min=180)
revenue = (qualified_leads * revenue_per_lead).round(2)

df = pd.DataFrame({
    "week": week,
    "week_start": week_start.strftime("%Y-%m-%d"),
    "followers": followers,
    "impressions": impressions,
    "paid_spend": paid_spend,
    "sessions": sessions,
    "qualified_leads": qualified_leads,
    "qualified_lead_rate": (qualified_leads / sessions).round(5),
    "revenue": revenue,
})

df.to_csv("vanity_metrics_weekly.csv", index=False)
df.head()

In [ ]:
print(f"Followers: {df.followers.iloc[0]:,} -> {df.followers.iloc[-1]:,} "
      f"({(df.followers.iloc[-1] / df.followers.iloc[0] - 1) * 100:.0f}%)")
print(f"Revenue, first 8 weeks:  ${df.revenue[:8].mean():,.0f} / week")
print(f"Revenue, weeks 20 to 28: ${df.revenue[19:28].mean():,.0f} / week")
print(f"Revenue, last 8 weeks:   ${df.revenue[-8:].mean():,.0f} / week")

## Part 3. The setup

Two charts, same 52 weeks. One of them noticed that the business lost a quarter
of its revenue.

In [ ]:
DIP_START, DIP_END = 17, 31

fig, axes = plt.subplots(2, 1, figsize=(7.5, 5.4), sharex=True)

ax = axes[0]
ax.axvspan(DIP_START, DIP_END, color=DIP_SHADE, zorder=0)
ax.plot(df.week, df.followers, color=VANITY, linewidth=2.0)
ax.set_ylabel("Social followers")
ax.set_title("The metric on the dashboard", loc="left")
ax.grid(axis="y", alpha=0.7); ax.set_axisbelow(True)
ax.annotate(f"+{(df.followers.iloc[-1]/df.followers.iloc[0]-1)*100:.0f}% over the year",
            xy=(46, df.followers.iloc[45]), xytext=(27, df.followers.min() + 250),
            color=VANITY, fontsize=9, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=VANITY, linewidth=1.0))

ax = axes[1]
ax.axvspan(DIP_START, DIP_END, color=DIP_SHADE, zorder=0)
rev_k = df.revenue / 1000
roll = rev_k.rolling(4, center=True, min_periods=1).mean()
ax.plot(df.week, rev_k, color=DECISION, linewidth=1.0, alpha=0.35, label="Weekly revenue")
ax.plot(df.week, roll, color=DECISION, linewidth=2.2, label="4-week average")
ax.set_ylabel("Revenue ($K / week)"); ax.set_xlabel("Week")
ax.set_title("The metric that pays the bills", loc="left")
ax.grid(axis="y", alpha=0.7); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", ncol=2, bbox_to_anchor=(0.0, 1.02))
ax.text((DIP_START + DIP_END) / 2, rev_k.max() * 0.80,
        "revenue fell ~50%\nthen recovered", color=DECISION,
        fontsize=9, fontweight="bold", ha="center", va="center")

fig.tight_layout()
fig.savefig("fig1_followers_vs_revenue.png")
plt.show()

## Part 4. Which metric actually tracked revenue?

Same revenue on both y-axes. The only thing that changes is what we put on the x-axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.9))

panels = [("followers", "Social followers", VANITY, "Vanity metric"),
          ("qualified_lead_rate", "Qualified lead rate", DECISION, "Decision metric")]

for ax, (col, label, color, tag) in zip(axes, panels):
    x, y = df[col].values, df.revenue.values / 1000
    r = np.corrcoef(x, y)[0, 1]
    ax.scatter(x, y, s=26, color=color, alpha=0.65,
               edgecolor="white", linewidth=0.5, zorder=3)
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, slope * xs + intercept, color=color,
            linewidth=1.6, linestyle="--", zorder=2)
    if col == "qualified_lead_rate":
        ax.set_xlabel(label + " (% of sessions)")
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v*100:.1f}%"))
    else:
        ax.set_xlabel(label)
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_ylabel("Revenue ($K / week)")
    ax.set_title(f"{tag}:  r = {r:+.2f}", loc="left", color=color)
    ax.grid(alpha=0.7); ax.set_axisbelow(True)

fig.suptitle("Same 52 weeks. Same revenue. Two very different stories.",
             fontsize=11, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("fig2_scatter_comparison.png")
plt.show()

Widening it out to all six metrics shows something more useful than two scatter
plots. The correlation matrix splits into blocks.

Vanity metrics are not random noise. They agree with **each other**, which is
exactly why a dashboard full of them feels like a real measurement system. They
just form a system that is disconnected from revenue.

In [ ]:
cols = ["followers", "impressions", "sessions",
        "qualified_lead_rate", "qualified_leads", "revenue"]
labels = ["Followers", "Impressions", "Sessions",
          "Qual. lead rate", "Qual. leads", "Revenue"]
corr = df[cols].corr().values

fig, ax = plt.subplots(figsize=(6.4, 5.4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=40, ha="right"); ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=8.5,
                color="white" if abs(corr[i, j]) > 0.55 else "#222222")

ax.axhline(1.5, color="#222222", linewidth=1.6)
ax.axvline(1.5, color="#222222", linewidth=1.6)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)

cbar = fig.colorbar(im, ax=ax, shrink=0.78)
cbar.set_label("Pearson correlation", fontsize=9)
cbar.outline.set_visible(False)
ax.set_title("Vanity metrics correlate with each other,\nnot with revenue", loc="left")

fig.tight_layout()
fig.savefig("fig3_correlation_heatmap.png")
plt.show()

print(df[cols].corr()["revenue"].sort_values(ascending=False).round(3))

## Part 5. The three-question test

This is the part to edit. Correlation needs a year of history, but you can
screen a metric in about ten seconds with three questions:

1. If this number **doubled** tomorrow, what would I do differently?
2. If it **halved** tomorrow, what would I do differently?
3. Does it move **before or after** money changes hands?

If questions 1 and 2 have the same answer, it is not a metric. It is a mood.

Replace the entries below with the numbers on your own dashboard.

In [ ]:
# Edit this list. Be honest on the first two fields.
my_metrics = [
    # name,                     action_if_doubled,      action_if_halved,        timing
    ("Instagram followers",     "post more",            "post more",             "after"),
    ("Impressions",             "nothing",              "nothing",               "after"),
    ("Sessions",               "nothing",               "check acquisition",     "before"),
    ("Qualified lead rate",     "raise budget",         "audit form + targeting", "before"),
    ("Cost per qualified lead", "scale the channel",    "pause the channel",      "before"),
]

print(f"{'Metric':<26} {'Verdict':<16} Why")
print("-" * 78)
for name, up, down, timing in my_metrics:
    same = up.strip().lower() == down.strip().lower()
    inert = up.strip().lower() in {"nothing", "", "none"}
    if same or inert:
        verdict, why = "VANITY", "same (or no) action either direction"
    elif timing == "after":
        verdict, why = "LAGGING", "actionable, but only tells you after the fact"
    else:
        verdict, why = "DECISION", "moves before money and changes what you do"
    print(f"{name:<26} {verdict:<16} {why}")

## What this notebook cannot tell you

Worth being straight about the limits, because a correlation of +0.88 is
persuasive and persuasive is dangerous.

**Fifty-two observations is not a lot.** These correlations are an illustration
of where to look first, not an identification strategy. Nothing here establishes
that lead rate *causes* revenue.

**The dip recovered on its own here.** In real life a metric that falls and comes
back is the classic setup for claiming credit for a fix that did nothing. That is
regression to the mean, and it deserves its own article.

**Followers are not worth zero.** They are worth whatever your follower-to-lead
rate says they are worth. The problem is that almost nobody has calculated that
number, which is what lets a follower count masquerade as a result.

For the causal question, see the articles on incrementality and lift linked in
the README.

---

**Joe Domaleski** | [Marketing Data Science](https://blog.marketingdatascience.ai)